### Imports

In [1]:
from experiments._common import H4_FCIDUMP_PATH, H6_FCIDUMP_PATH

from qiskit.quantum_info import SparsePauliOp
import numpy as np

from qcmps import (
    FCIDumpLoader,
    GlobalQCMPS, Ansatz, HFPreset,
    Logger,                      
    GlobalOptimizer, OneSiteOptimizer, TwoSiteOptimizer,
    SO4Gate, BrickWallLayerType,
)

### Logger

In [2]:
name = "h4_test"

logging_path = f"{name}.log"
logger = Logger(logging_path)

### Building the system

In [3]:
loader = FCIDumpLoader(H4_FCIDUMP_PATH)
loader.interleave_spin_orbitals()

nuclear_repulsion = loader.nuclear_repulsion

hf_occupancy = loader.hf_occupancy()

num_phys_qubits = loader.n_spin_orbitals
num_bond_qubits = 2
num_qubits = num_phys_qubits + num_bond_qubits

full_hamiltonian = SparsePauliOp.from_list([
    ("I" * num_bond_qubits + label, coefficient)
    for label, coefficient in loader.hamiltonian.to_list()
])

In [4]:
ansatz = Ansatz(num_qubits)

hf_preset = HFPreset(num_phys_qubits, hf_occupancy)
bw_layer = BrickWallLayerType(SO4Gate)
qcmps = GlobalQCMPS(num_phys_qubits, num_bond_qubits, [bw_layer])

ansatz.add_component(hf_preset, list(range(num_bond_qubits, num_qubits)))
ansatz.add_component(qcmps, list(range(num_qubits)))

ansatz.add_hamiltonian(full_hamiltonian, list(range(num_qubits)))
ansatz.add_nuclear_repulsion(nuclear_repulsion)

### Loading the weights

In [ ]:
checkpoint_path = "h4_q2_seed0_layer4_continue.npz"

with np.load(checkpoint_path) as data:
    qcmps.update_parameters(data["parameters"])

### Initial energy

In [5]:
logger.log(f"Energy: {ansatz.total_energy():.12f}")


Energy: -2.215215410025


### Uniform parameters

In [6]:
qcmps.uniform_parameters()
logger.log(f"Initial energy: {ansatz.total_energy():.12f}")

Initial energy: -0.185286952122


### Global optimization

In [ ]:
logger.log("Starting Global optimization...")
global_optimizer = GlobalOptimizer(logger)
result = global_optimizer.optimize(ansatz, qcmps, options = {"maxiter": 5, "maxfun": 100_000, "gtol": 1e-7, "ftol": 1e-12})

logger.log(f"Final energy: {ansatz.total_energy():.12f}")

### Adding layers

In [ ]:
logger.log(f"Adding additional layers to QCMPS...")
qcmps.add_layers([bw_layer])
ansatz.rebuild()
logger.log(f"Energy after adding layers: {ansatz.total_energy():.12f}")

### Perturbing parameters

In [ ]:
qcmps.perturb_parameters(0.001)
logger.log(f"Energy after perturbing parameters: {ansatz.total_energy():.12f}")

### One site optimization

In [ ]:
logger.log("Starting One site optimization...")
global_optimizer = OneSiteOptimizer(logger, log_every_n_steps = 20, max_sweeps=5, local_maxiter = 10, tolerance = 1e-8)
result = global_optimizer.optimize(ansatz, qcmps)

logger.log(f"Final energy: {ansatz.total_energy():.12f}")

### Two site optimization

In [ ]:
logger.log("Starting Two site optimization...")
global_optimizer = TwoSiteOptimizer(logger, log_every_n_steps = 20, max_sweeps=2, local_maxiter=10, tolerance = 1e-8)
result = global_optimizer.optimize(ansatz, qcmps)

logger.log(f"Final energy: {ansatz.total_energy():.12f}")

### Saving the ansatz into checkpoint

In [ ]:
import json
from pathlib import Path

np.savez_compressed(
    f"{name}.npz",
    parameters=qcmps.current_parameters(),
)

record = {
    "system": "H4",
    "n_bond_qubits": num_bond_qubits,
    "bond_dimension": 2**num_bond_qubits,
    "n_layers": len(qcmps.layer_types),
    "n_parameters": qcmps.n_parameters,
    "total_energy": ansatz.total_energy(),
}
with Path(f"{name}.json").open("w", encoding="utf-8") as file:
    json.dump(record, file, indent=2)